# RetailMind AI — Bank Customer Churn Classification
## End-to-end project notebook

### Project flow

1. Inspect the data and understand the target.
2. Split into train and test sets before fitting any preprocessing.
3. Put imputation, encoding and scaling inside Scikit-learn pipelines.
4. Establish a Logistic Regression baseline.
5. Compare Decision Tree and Random Forest candidates.
6. Tune hyperparameters using cross-validation on training data only.
7. Check class imbalance and compare class-weight handling.
8. Test feature selection as an experiment.
9. Evaluate the selected pipeline once on the untouched test set.
10. Persist and reload the complete pipeline with Joblib.

**Main leakage rule:** split first. Fit preprocessing and feature selection on training data only. When these steps are inside a `Pipeline`, Scikit-learn fits them on each training fold during cross-validation and applies the learned transformations to the corresponding validation fold. The final fitted pipeline then transforms the test data automatically during `predict()`.

## 0. Project structure and running in VS Code

Open the **project root folder** in VS Code, not just the notebook file.

```text
RetailMind_Bank_Churn_Project/
│
├── data/
│   └── churn.csv                    # Raw dataset (not committed to Git by default)
│
├── notebooks/
│   └── Bank_Churn_End_to_End.ipynb  # This notebook
│
├── src/                             # Reusable application code added later
│   └── .gitkeep
│
├── models/                          # Fitted model/pipeline artifacts
│   └── .gitkeep
│
├── reports/                         # Metrics, comparison tables and plots
│   └── .gitkeep
│
├── README.md
├── requirements.txt
└── .gitignore
```

### Run in VS Code

1. Extract the project ZIP.
2. Open the extracted `RetailMind_Bank_Churn_Project` folder in VS Code.
3. Install the **Python** and **Jupyter** extensions.
4. Open a terminal in the project root.
5. Create and activate an environment if you want an isolated setup:

```bash
python -m venv .venv
```

Windows PowerShell:

```powershell
.venv\Scripts\Activate.ps1
```

6. Install dependencies:

```bash
python -m pip install -r requirements.txt
```

7. Open `notebooks/Bank_Churn_End_to_End.ipynb`.
8. Select the Python kernel from the environment where you installed the packages.
9. Run cells one at a time with **Shift + Enter**, or use **Run All** after checking the dataset path.

### Dataset placement

Place the CSV from your original notebook in `data/`. The setup cell accepts `churn.csv`, `Churn_Modelling.csv`, or another single `.csv` file in that folder. The file must contain the target column `Exited`.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 8
TARGET = "Exited"

print("Libraries imported.")

In [ ]:
# Locate the project folders relative to this notebook.
# Expected location: project_root/notebooks/Bank_Churn_End_to_End.ipynb

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
SRC_DIR = PROJECT_ROOT / "src"

for folder in [DATA_DIR, MODEL_DIR, REPORT_DIR, SRC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Find the dataset without hard-coding a machine-specific path such as E:/data/churn.csv.
preferred_names = ["churn.csv", "Churn_Modelling.csv", "Churn_Modelling (1).csv"]
DATA_PATH = next(
    (DATA_DIR / name for name in preferred_names if (DATA_DIR / name).exists()),
    None
)

if DATA_PATH is None:
    csv_files = list(DATA_DIR.glob("*.csv"))
    if len(csv_files) == 1:
        DATA_PATH = csv_files[0]
    elif len(csv_files) == 0:
        raise FileNotFoundError(
            f"No CSV found in {DATA_DIR}. Copy your bank churn CSV into the data folder."
        )
    else:
        raise FileNotFoundError(
            f"More than one CSV found in {DATA_DIR}. Set DATA_PATH explicitly."
        )

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH.name)
print("Models directory:", MODEL_DIR)
print("Reports directory:", REPORT_DIR)

# 1. Load and understand the data

The uploaded notebook uses the bank churn dataset. Its target is `Exited`:
- `0` = customer did not exit;
- `1` = customer exited.

The original notebook selected features using `df.iloc[:, 2:-1]`. That relies on column positions and can silently select the wrong fields if the CSV column order changes. We will instead remove identifier columns by name and select the target explicitly.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nData types and non-null counts:")
df.info()

In [ ]:
# Summary statistics for numeric columns.
display(df.describe().T)

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nTarget counts:")
display(df[TARGET].value_counts(dropna=False))

print("\nTarget proportions:")
display(df[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# Preserve the original notebook's Geography vs. churn exploration.
if "Geography" in df.columns:
    display(df.groupby("Geography")[TARGET].mean().sort_values(ascending=False))

## 1.1 Understand the class imbalance

A model that predicts `Exited = 0` for every customer can therefore achieve deceptively high accuracy while detecting no customers who actually exit.

We should report more than accuracy:
- **Recall for class 1:** among customers who exited, how many did we identify?
- **Precision for class 1:** among customers flagged as likely to exit, how many actually exited?
- **F1-score:** balance between precision and recall.
- **ROC-AUC:** how well the model ranks positive cases above negative cases across thresholds.

For a retention use case, recall is important, but very low precision can create too many unnecessary retention interventions.

In [ ]:
majority_class = df[TARGET].value_counts(normalize=True).idxmax()
majority_baseline_accuracy = (df[TARGET] == majority_class).mean()

print("Majority class:", majority_class)
print("Majority-class baseline accuracy:", round(majority_baseline_accuracy, 4))

ax = df[TARGET].value_counts().sort_index().plot(
    kind="bar", rot=0, title="Customer exit distribution"
)
ax.set_xlabel("Exited (0 = No, 1 = Yes)")
ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

# 2. Prepare features and split the data

First remove identifiers such as `RowNumber`, `CustomerId`, and `Surname`. These identify a row or person; they are not meaningful behavioural predictors for this model.

Then split the data **before** fitting imputers, encoders, scalers, feature selectors, or dimensionality reduction.

`stratify=y` keeps the proportion of exited and retained customers approximately consistent in both subsets.

In [ ]:
# Drop identifiers only when they are present.
identifier_columns = ["RowNumber", "CustomerId", "Surname"]
X = df.drop(columns=[TARGET] + [c for c in identifier_columns if c in df.columns])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

print("\nExited rate — training:", round(y_train.mean(), 3))
print("Exited rate — test:    ", round(y_test.mean(), 3))

# 3. Preprocessing — fit on training data, apply to test data

The dataset has numeric features (for example `CreditScore`, `Age`, `Balance`) and categorical features (`Geography`, `Gender`).

We need different transformations for these two groups:

| Feature type | Transformation | Why |
|---|---|---|
| Numeric | Median imputation | Handles missing numeric values |
| Numeric for Logistic Regression | StandardScaler | Puts features on comparable scales |
| Categorical | Most-frequent imputation | Handles missing categories |
| Categorical | OneHotEncoder | Converts categories into numeric indicator columns |

We identify the column names from `X_train`, then define a `ColumnTransformer`. This is a standard Scikit-learn pattern.

### Why not fit on all of X?

If the scaler learns the mean and standard deviation from the full dataset, it has indirectly used information from the test set. The test set is meant to simulate future unseen customers.

By placing preprocessing inside a Pipeline:
- `.fit(X_train, y_train)` learns imputation values, category mappings and scaling values from training data;
- `.predict(X_test)` applies those already-learned transformations to the test data;
- during cross-validation, each fold gets its own preprocessing fit using only that fold's training portion.

In [ ]:
# Determine feature groups from the training data only.
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 3.1 Define the preprocessing steps

We create two preprocessors:
- `preprocessor_scaled` for Logistic Regression;
- `preprocessor_tree` for Decision Tree and Random Forest.

Tree models generally do not need standardisation because their splits depend on feature thresholds, not distance or coefficient magnitude. Scaling is useful for Logistic Regression because its optimisation is sensitive to feature scales.

The `OneHotEncoder` is configured with `handle_unknown="ignore"` so an unfamiliar category in future data does not cause prediction to fail.

In [ ]:
numeric_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ("numeric", numeric_scaled, numeric_features),
    ("categorical", categorical_preprocessing, categorical_features),
])

preprocessor_tree = ColumnTransformer(transformers=[
    ("numeric", numeric_unscaled, numeric_features),
    ("categorical", categorical_preprocessing, categorical_features),
])

print("Preprocessing objects defined. They have not been fitted yet.")

# 4. Day 1 — Logistic Regression baseline

Logistic Regression estimates the probability of a binary outcome. It models the log-odds of the positive class as a linear combination of the input features, then converts that value into a probability using the sigmoid function.

The important implementation detail is that preprocessing and the classifier are in **one Pipeline**. We pass the original, unscaled `X_train` to `.fit()`. We do not manually scale the test set.

### Logistic Regression intuition

Logistic Regression first calculates a linear score from the input features, then converts that score into a probability between 0 and 1 using the sigmoid function.

The model is linear in the **log-odds**:

`log(p / (1 - p)) = b0 + b1*x1 + b2*x2 + ... + bn*xn`

where `p` is the probability that `Exited = 1`. The default prediction threshold is 0.5, although a business may later choose another threshold based on the cost of false negatives and false positives.

In [ ]:
logistic_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor_scaled)),
    ("model", LogisticRegression(max_iter=20000, random_state=RANDOM_STATE)),
])

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression baseline fitted.")

## 4.1 Evaluate with consistent metrics

The following helper only calculates and displays metrics. It does not fit or transform anything, so it does not create data leakage.

The confusion matrix layout is:

```text
                 Predicted 0   Predicted 1
Actual 0             TN            FP
Actual 1             FN            TP
```

In [ ]:
def evaluate_model(name, fitted_pipeline, X_eval, y_eval, show_report=True):
    """Evaluate a fitted pipeline; positive class is Exited=1."""
    predictions = fitted_pipeline.predict(X_eval)
    probabilities = fitted_pipeline.predict_proba(X_eval)[:, 1]

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1": f1_score(y_eval, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_eval, probabilities),
    }

    if show_report:
        display(pd.DataFrame([results]).set_index("Model"))
        print("Confusion matrix [[TN, FP], [FN, TP]]:")
        print(confusion_matrix(y_eval, predictions))
        print("\nClassification report:")
        print(classification_report(
            y_eval, predictions,
            target_names=["Retained (0)", "Exited (1)"],
            zero_division=0,
        ))

    return results


day1_metrics = evaluate_model(
    "Day 1 — Logistic Regression",
    logistic_pipeline,
    X_test,
    y_test,
)

In [ ]:
# Keep the original baseline artifact so later model generations can be compared.
day1_model_path = MODEL_DIR / "bank_churn_day1_logistic_baseline.joblib"
joblib.dump(logistic_pipeline, day1_model_path)
print("Day 1 baseline saved to:", day1_model_path)

In [ ]:
# ROC curve uses predicted probabilities, not predicted class labels.
RocCurveDisplay.from_estimator(logistic_pipeline, X_test, y_test)
plt.title("Day 1 — Logistic Regression ROC curve")
plt.tight_layout()
plt.show()

# 5. Day 2 — Decision Tree and Random Forest

We now compare the two tree-based algorithms from the original notebook.

- **Decision Tree:** a single sequence of feature-based decisions. It is easy to visualise but can overfit.
- **Random Forest:** many decision trees trained on different samples/features; their predictions are combined to improve stability.

The preprocessing pipeline is reused, but the tree preprocessor does not scale numeric variables because trees do not require it.

In [ ]:
decision_tree_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=5,
        min_samples_leaf=10,
        max_leaf_nodes=10,
        min_samples_split=5,
    )),
])

random_forest_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])

decision_tree_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)

print("Decision Tree and Random Forest fitted.")

In [ ]:
# Use stratified folds so each fold preserves the class proportions.
# This is defined here because Day 2 uses cross-validation to select its champion.
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
print("Cross-validation strategy ready.")

In [ ]:
# Compare candidates using CV on training data, then evaluate on the same
# untouched test set. CV metrics—not test metrics—determine the champion.
day2_candidates = {
    "Logistic Regression": clone(logistic_pipeline),
    "Decision Tree": clone(decision_tree_pipeline),
    "Random Forest": clone(random_forest_pipeline),
}

day2_fitted = {}
day2_rows = []

for model_name, candidate_pipeline in day2_candidates.items():
    cv_result = cross_validate(
        candidate_pipeline,
        X_train,
        y_train,
        scoring={"recall": "recall", "precision": "precision"},
        cv=cv_strategy,
        n_jobs=-1,
        return_train_score=False,
    )

    candidate_pipeline.fit(X_train, y_train)
    day2_fitted[model_name] = candidate_pipeline

    test_metrics = evaluate_model(
        f"Day 2 — {model_name}",
        candidate_pipeline,
        X_test,
        y_test,
        show_report=False,
    )
    test_metrics["CV_Recall"] = cv_result["test_recall"].mean()
    test_metrics["CV_Precision"] = cv_result["test_precision"].mean()
    day2_rows.append(test_metrics)

day2_comparison = pd.DataFrame(day2_rows).sort_values(
    "CV_Recall", ascending=False
)
display(day2_comparison)
day2_comparison.to_csv(REPORT_DIR / "day2_model_comparison.csv", index=False)

day2_champion_name = day2_comparison.iloc[0]["Model"].replace("Day 2 — ", "")
day2_champion = day2_fitted[day2_champion_name]

day2_champion_path = MODEL_DIR / "bank_churn_day2_champion.joblib"
joblib.dump(day2_champion, day2_champion_path)

print("Day 2 champion:", day2_champion_name)
print("Selection basis: mean cross-validation recall on training data.")
print("Saved:", day2_champion_path)

In [ ]:
# Visualise the Decision Tree.
# This is a plot of the fitted tree after the preprocessing transformation.
feature_names_after_encoding = (
    decision_tree_pipeline.named_steps["preprocess"].get_feature_names_out()
)

plt.figure(figsize=(20, 10))
plot_tree(
    decision_tree_pipeline.named_steps["model"],
    feature_names=feature_names_after_encoding,
    class_names=["Retained", "Exited"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)
plt.title("Decision Tree — first levels")
plt.tight_layout()
plt.show()

# 6. Day 3 — Cross-validation and hyperparameter tuning

A single train/test split can give a result that depends on that particular split. Cross-validation provides a more stable estimate using several training/validation folds.

**Important separation:**
- `GridSearchCV` sees only `X_train` and `y_train`.
- Each fold fits its own preprocessing and model.
- The test set remains untouched until final evaluation.

We will tune the Decision Tree and Random Forest. The scoring metric is **recall for `Exited = 1`**, because the business wants to identify customers who may leave. We will still report precision, F1 and ROC-AUC to understand the trade-off.

In [ ]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

decision_tree_grid = {
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 5, 10],
    "model__class_weight": [None, "balanced"],
}

random_forest_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 3],
    "model__class_weight": [None, "balanced"],
}

print("Cross-validation strategy and search grids are ready.")

In [ ]:
# Tune Decision Tree.
dt_search_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

dt_search = GridSearchCV(
    estimator=dt_search_pipeline,
    param_grid=decision_tree_grid,
    scoring="recall",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

dt_search.fit(X_train, y_train)

print("Best Decision Tree CV recall:", round(dt_search.best_score_, 4))
print("Best parameters:", dt_search.best_params_)

In [ ]:
# Tune Random Forest.
rf_search_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
])

rf_search = GridSearchCV(
    estimator=rf_search_pipeline,
    param_grid=random_forest_grid,
    scoring="recall",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

rf_search.fit(X_train, y_train)

print("Best Random Forest CV recall:", round(rf_search.best_score_, 4))
print("Best parameters:", rf_search.best_params_)

In [ ]:
# Compare the best cross-validation results.
tuning_comparison = pd.DataFrame([
    {
        "Model": "Decision Tree",
        "Best_CV_Recall": dt_search.best_score_,
        "Best_Parameters": str(dt_search.best_params_),
    },
    {
        "Model": "Random Forest",
        "Best_CV_Recall": rf_search.best_score_,
        "Best_Parameters": str(rf_search.best_params_),
    },
]).sort_values("Best_CV_Recall", ascending=False)

display(tuning_comparison)
tuning_comparison.to_csv(REPORT_DIR / "day3_tuning_comparison.csv", index=False)

# Champion is selected from CV results on training data.
if dt_search.best_score_ >= rf_search.best_score_:
    champion_name = "Tuned Decision Tree"
    champion_pipeline = dt_search.best_estimator_
else:
    champion_name = "Tuned Random Forest"
    champion_pipeline = rf_search.best_estimator_

print("Current champion:", champion_name)

# Save this tuned model generation separately from the Day 1 and Day 2 artifacts.
day3_model_path = MODEL_DIR / "bank_churn_day3_tuned_champion.joblib"
joblib.dump(champion_pipeline, day3_model_path)
print("Day 3 tuned champion saved to:", day3_model_path)

## 6.1 Evaluate the selected champion on the hold-out test set

This is the first point at which the selected champion is evaluated on the test set. Do not use this test result to go back and tune parameters; doing so would turn the test set into part of model selection.

In [ ]:
champion_test_metrics = evaluate_model(
    champion_name,
    champion_pipeline,
    X_test,
    y_test,
)

# 7. Class imbalance — is it worth handling?

The target is imbalanced: fewer customers exited than stayed. Accuracy alone can hide poor detection of the minority class.

Scikit-learn's `class_weight="balanced"` increases the penalty for mistakes on the less frequent class. It is a sensible first experiment for this dataset, but it is not guaranteed to improve every metric.

We included `class_weight` in the hyperparameter grids above. The selected `best_params_` tells us whether cross-validation preferred `None` or `"balanced"` for each model.

Do not apply SMOTE or oversampling before the train/test split. If oversampling is explored later, it must happen **inside each training fold** using an imbalanced-learn Pipeline. Never resample the test set.

In [ ]:
print("Decision Tree selected class_weight:",
      dt_search.best_params_.get("model__class_weight"))
print("Random Forest selected class_weight:",
      rf_search.best_params_.get("model__class_weight"))

print("\nClass distribution in the training data:")
display(y_train.value_counts().rename(index={0: "Retained", 1: "Exited"}))

# 8. Feature selection — an experiment, not an automatic requirement

The original dataset has a modest number of useful input columns. Feature selection may reduce noise or simplify the model, but it can also remove useful signal.

We will test `SelectKBest` inside the pipeline. This is important: the selector must be fitted separately within each cross-validation training fold. If we select features once using the full dataset before cross-validation, information leaks into validation folds.

We will compare:
- the tuned champion without feature selection;
- the same model family with feature selection.

The `k` values below refer to the transformed features after categorical one-hot encoding.

In [ ]:
# Rebuild the same model family with a feature-selection step.
# Use the estimator settings selected by GridSearchCV.
if champion_name == "Tuned Decision Tree":
    selected_estimator = clone(dt_search.best_estimator_.named_steps["model"])
else:
    selected_estimator = clone(rf_search.best_estimator_.named_steps["model"])

feature_selection_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("select", SelectKBest(score_func=f_classif)),
    ("model", selected_estimator),
])

feature_selection_grid = {
    "select__k": [5, 8, 10],
}

feature_search = GridSearchCV(
    estimator=feature_selection_pipeline,
    param_grid=feature_selection_grid,
    scoring="recall",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

feature_search.fit(X_train, y_train)

print("Best feature-selection CV recall:", round(feature_search.best_score_, 4))
print("Best parameters:", feature_search.best_params_)

In [ ]:
# Compare cross-validation recall before deciding whether to keep feature selection.
feature_comparison = pd.DataFrame([
    {
        "Approach": f"{champion_name} — all transformed features",
        "CV_Recall": max(dt_search.best_score_, rf_search.best_score_),
    },
    {
        "Approach": f"{champion_name} — SelectKBest",
        "CV_Recall": feature_search.best_score_,
    },
])

display(feature_comparison)

# 9. Dimensionality reduction — should we use PCA?

**PCA is not included in the main workflow because it is not clearly necessary here.**

Reasons:
- The original bank dataset has a relatively small number of input features.
- After one-hot encoding, the feature count is still manageable.
- Tree models do not generally need PCA.
- PCA creates combinations of original variables, which makes business interpretation harder.
- Feature selection is a more direct experiment when the goal is to retain a subset of meaningful input features.

PCA becomes more attractive when there are many highly correlated numeric variables or a very high-dimensional feature space. If it is tested, it must also be placed inside the Pipeline so it is fitted only on each training fold.

For this project, compare the original features with SelectKBest and document whether feature selection helped. Do not add PCA simply to include another technique.

# 10. Final model choice and final test evaluation

Choose between the tuned champion and the feature-selection candidate using their cross-validation recall. Only after this choice do we use the hold-out test set for the final evaluation.

This keeps the test set independent from model selection.

In [ ]:
if feature_search.best_score_ > max(dt_search.best_score_, rf_search.best_score_):
    final_model_name = f"{champion_name} + SelectKBest"
    final_pipeline = feature_search.best_estimator_
else:
    final_model_name = champion_name
    final_pipeline = champion_pipeline

print("Final model selected using training CV:", final_model_name)

final_test_metrics = evaluate_model(
    final_model_name,
    final_pipeline,
    X_test,
    y_test,
)

# 11. Save and reload the complete pipeline with Joblib

## What is Joblib?

`joblib` is a Python library commonly used to persist Scikit-learn models, especially when they contain NumPy arrays.

- `joblib.dump(model, path)` serialises the fitted object and writes it to disk.
- `joblib.load(path)` reads the saved object back into Python.

We save the **entire fitted Pipeline**, not just the classifier. That matters because the fitted pipeline contains:
- the imputation values learned from training data;
- the category mappings learned by OneHotEncoder;
- the scaling parameters (for the Logistic Regression pipeline, if selected);
- any feature-selection state;
- the fitted estimator.

When new customer rows arrive, call `.predict()` on the loaded pipeline with the original input columns. The pipeline applies the exact preprocessing steps in the correct order.

**Security note:** Joblib uses Python object serialisation. Only load model files from trusted sources. A model file may also depend on compatible versions of scikit-learn and other libraries.

In [ ]:
model_path = MODEL_DIR / "bank_churn_champion.joblib"

joblib.dump(final_pipeline, model_path)
print("Saved model to:", model_path)

# Load the persisted pipeline.
loaded_pipeline = joblib.load(model_path)
print("Model loaded successfully.")

# Confirm that saving and loading did not change predictions.
predictions_before_save = final_pipeline.predict(X_test)
predictions_after_load = loaded_pipeline.predict(X_test)

print(
    "Predictions identical after reload:",
    np.array_equal(predictions_before_save, predictions_after_load)
)

In [ ]:
# Example: score a few customer rows using the loaded pipeline.
sample_customers = X_test.head(5)

sample_predictions = loaded_pipeline.predict(sample_customers)
sample_probabilities = loaded_pipeline.predict_proba(sample_customers)[:, 1]

prediction_output = sample_customers.copy()
prediction_output["predicted_exited"] = sample_predictions
prediction_output["probability_of_exit"] = sample_probabilities

display(prediction_output)

# 12. What belongs in each project directory?

| Directory | What belongs here | Example |
|---|---|---|
| `data/` | Raw input datasets and, if needed, cleaned data extracts | `churn.csv` |
| `notebooks/` | Exploratory work, experiments and documented analysis | `Bank_Churn_End_to_End.ipynb` |
| `src/` | Reusable Python modules used by scripts or services | `preprocessing.py`, `predict.py`, `train.py` |
| `models/` | Persisted fitted models and complete pipelines | `bank_churn_champion.joblib` |
| `reports/` | Model metrics, comparison tables, plots and experiment summaries | `day3_tuning_comparison.csv`, `roc_curve.png` |

## What is `.gitkeep`?

Git does not track empty directories. A file named `.gitkeep` is a convention used to make an otherwise empty directory appear in Git.

It is not a special Git command or file type. It can be an empty file. Once a real file is added to that directory, the `.gitkeep` file can be removed if you want.

The `.gitignore` file controls which files Git should ignore. In this project, the raw CSV and generated model binaries are ignored by default. This avoids accidentally committing large or restricted files. Decide separately whether reports and model artifacts should be versioned or stored in an artifact repository.

# 13. Git workflow

Run these commands from the project root:

```bash
git init
git status
git add README.md requirements.txt .gitignore notebooks/
git add reports/
git commit -m "Build bank customer churn classification workflow"
```

Because `data/*.csv` and `models/*.joblib` are ignored in the supplied `.gitignore`, they will not be added by these commands. That is intentional by default:
- raw data may be licensed or too large to commit;
- fitted model binaries are often better stored in an artifact store or managed model registry;
- the notebook, code, requirements and reports make the experiment reproducible.

## Final workflow recap

```text
Raw CSV
  ↓
Train/test split
  ↓
Pipeline: imputation → encoding → scaling (where appropriate)
  ↓
Model training and cross-validation on training data
  ↓
Hyperparameter tuning / class-weight experiment / feature selection
  ↓
Select champion using CV results
  ↓
One final evaluation on untouched test data
  ↓
Save complete pipeline with Joblib
  ↓
Reload and predict on new customer records
```

**This is a sound Scikit-learn project workflow.** It introduces preprocessing, leakage prevention, model comparison, tuning, class-imbalance handling and feature selection without forcing PCA or oversampling where they are not yet justified.